In [10]:
import pandas as pd
import numpy as np
from scipy import stats

In [11]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results" / "esda"
OUTPUT_DIR = here() / "results" / "validation"

In [14]:
df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_index.csv", dtype={'CVEGEO': str})

In [15]:
df["CVEGEO"] = df["CVEGEO"].str.zfill(5)
df["rank_RELMS"] = df["rank"]
df["rank_RELM"] = df["RELM_index"].rank(ascending=False, method="min").astype(int)
df["rank_diff"] = df["rank_RELMS"] - df["rank_RELM"]

### Spearman & Kendall rank correlation

In [16]:
rho, rho_p = stats.spearmanr(df["rank_RELMS"], df["rank_RELM"])
tau, tau_p = stats.kendalltau(df["rank_RELMS"], df["rank_RELM"])

print("=" * 60)
print("RANK CORRELATION")
print("=" * 60)
print(f"Spearman's rho = {rho:.4f}  (p = {rho_p:.3e})")
print(f"Kendall's tau  = {tau:.4f}  (p = {tau_p:.3e})")

RANK CORRELATION
Spearman's rho = 0.8639  (p = 0.000e+00)
Kendall's tau  = 0.7171  (p = 0.000e+00)


### Wilcoxon signed-rank test on paired index values

In [18]:
wilcoxon_stat, wilcoxon_p = stats.wilcoxon(df["RELMS_index"], df["RELM_index"])

diff = df["RELMS_index"] - df["RELM_index"]
n = len(diff)
# rank-biserial effect size for Wilcoxon signed-rank
ranks = stats.rankdata(np.abs(diff))
r_plus = ranks[diff > 0].sum()
r_minus = ranks[diff < 0].sum()
rank_biserial = (r_plus - r_minus) / (r_plus + r_minus)

print()
print("=" * 60)
print("WILCOXON SIGNED-RANK TEST (RELMS_index vs RELM_index)")
print("=" * 60)
print(f"W = {wilcoxon_stat:.2f}, p = {wilcoxon_p:.3e}")
print(f"Rank-biserial r = {rank_biserial:.4f}")
print(f"Mean(RELMS - RELM) = {diff.mean():.4f}, Median = {diff.median():.4f}")


WILCOXON SIGNED-RANK TEST (RELMS_index vs RELM_index)
W = 411762.00, p = 1.541e-218
Rank-biserial r = 0.7319
Mean(RELMS - RELM) = 0.0358, Median = 0.0498


### Top-N overlap (Jaccard index) between RELMS and RELM rankings

In [22]:
print()
print("=" * 60)
print("TOP-N OVERLAP (Jaccard index)")
print("=" * 60)

top_n_values = [10, 20, 50, 100]
# also percentage-based cuts
pct_values = [0.05, 0.10, 0.20]

overlap_results = []

for n_top in top_n_values:
    top_relms = set(df.nsmallest(n_top, "rank_RELMS")["CVEGEO"])
    top_relm = set(df.nsmallest(n_top, "rank_RELM")["CVEGEO"])
    intersection = len(top_relms & top_relm)
    union = len(top_relms | top_relm)
    jaccard = intersection / union if union > 0 else np.nan
    overlap_results.append({
        "cutoff": f"Top-{n_top}",
        "n": n_top,
        "n_shared": intersection,
        "jaccard": jaccard,
        "pct_overlap": intersection / n_top,
    })
    print(f"Top-{n_top:>4}: {intersection}/{n_top} shared "
          f"({intersection/n_top:.1%} overlap), Jaccard = {jaccard:.4f}")

print()
for pct in pct_values:
    n_top = int(round(len(df) * pct))
    top_relms = set(df.nsmallest(n_top, "rank_RELMS")["CVEGEO"])
    top_relm = set(df.nsmallest(n_top, "rank_RELM")["CVEGEO"])
    intersection = len(top_relms & top_relm)
    union = len(top_relms | top_relm)
    jaccard = intersection / union if union > 0 else np.nan
    overlap_results.append({
        "cutoff": f"Top-{pct:.0%}",
        "n": n_top,
        "n_shared": intersection,
        "jaccard": jaccard,
        "pct_overlap": intersection / n_top,
    })
    print(f"Top-{pct:.0%} (n={n_top}): {intersection}/{n_top} shared "
          f"({intersection/n_top:.1%} overlap), Jaccard = {jaccard:.4f}")


TOP-N OVERLAP (Jaccard index)
Top-  10: 2/10 shared (20.0% overlap), Jaccard = 0.1111
Top-  20: 6/20 shared (30.0% overlap), Jaccard = 0.1765
Top-  50: 18/50 shared (36.0% overlap), Jaccard = 0.2195
Top- 100: 32/100 shared (32.0% overlap), Jaccard = 0.1905

Top-5% (n=124): 46/124 shared (37.1% overlap), Jaccard = 0.2277
Top-10% (n=248): 116/248 shared (46.8% overlap), Jaccard = 0.3053
Top-20% (n=496): 320/496 shared (64.5% overlap), Jaccard = 0.4762


In [26]:
overlap_df = pd.DataFrame(overlap_results)
# overlap_df.to_csv("/home/claude/relms_analysis/topN_overlap.csv", index=False)

summary_stats = pd.DataFrame([{
    "spearman_rho": rho, "spearman_p": rho_p,
    "kendall_tau": tau, "kendall_p": tau_p,
    "wilcoxon_W": wilcoxon_stat, "wilcoxon_p": wilcoxon_p,
    "rank_biserial_r": rank_biserial,
    "mean_score_diff": diff.mean(), "median_score_diff": diff.median(),
}])
# summary_stats.to_csv("/home/claude/relms_analysis/rank_comparison_summary.csv", index=False)

# Also save full per-municipality rank comparison, sorted by biggest rank shift
rank_detail = df[["CVEGEO", "rank_RELMS", "rank_RELM", "rank_diff",
                   "RELMS_index", "RELM_index", "idx_security", "crime"]].copy()
rank_detail["abs_rank_diff"] = rank_detail["rank_diff"].abs()
rank_detail = rank_detail.sort_values("abs_rank_diff", ascending=False)
# rank_detail.to_csv("/home/claude/relms_analysis/rank_detail_by_shift.csv", index=False)

print()
print("=" * 60)
print("TOP 10 MUNICIPALITIES BY RANK SHIFT MAGNITUDE")
print("=" * 60)
print(rank_detail.head(10).to_string(index=False))

print()
print("Saved: topN_overlap.csv, rank_comparison_summary.csv, rank_detail_by_shift.csv")


TOP 10 MUNICIPALITIES BY RANK SHIFT MAGNITUDE
CVEGEO  rank_RELMS  rank_RELM  rank_diff  RELMS_index  RELM_index  idx_security    crime  abs_rank_diff
 11020        2459         24       2435     0.006962    0.635976      0.000000 0.000000           2435
 24028        2318         49       2269     0.344918    0.611143      0.034995 0.034995           2269
 21114        2264          8       2256     0.358891    0.664520      0.030534 0.030534           2256
 22014        2413        171       2242     0.306193    0.560725      0.027226 0.027226           2242
 08037        2289        139       2150     0.353827    0.570834      0.052229 0.052229           2150
 23005        2222         83       2139     0.370911    0.593827      0.056456 0.056456           2139
 01001        2040         35       2005     0.396254    0.623750      0.064539 0.064539           2005
 14039        2389        467       1922     0.318990    0.497870      0.053756 0.053756           1922
 09015        194